# Automatic Differentiation

This notebook explores JAX's automatic differentiation capabilities, which provide exact derivatives through code transformation rather than numerical approximations.

### ✅ Exercise 41 — Gradient of a Scalar Function

`jax.grad` only works on functions that return a scalar (single number). It returns a new function — the derivative.

In [1]:
# Import JAX libraries
import jax
import jax.numpy as jnp

# Define a simple function and its gradient
f = lambda x: x ** 2
grad_f = jax.grad(f)
grad_f(3.0)  # → 6.0

Array(6., dtype=float32, weak_type=True)

### ✅ Exercise 42 — Gradient of a Multi-Term Polynomial

Tests that JAX correctly applies the chain rule across multiple terms automatically.

In [2]:
# Multi-term polynomial: f(x) = 3x³ - 2x² + x - 5
f = lambda x: 3*x**3 - 2*x**2 + x - 5
grad_f = jax.grad(f)
grad_f(2.0)  # f'(x) = 9x² - 4x + 1 → 9(4) - 4(2) + 1 = 29.0

Array(29., dtype=float32, weak_type=True)

### ✅ Exercise 43 — Value and Gradient Together

`value_and_grad` computes both function value and gradient in one forward pass — more efficient than separate calls.

In [3]:
# Compute both value and gradient efficiently
f = lambda x: jnp.sin(x)
val, grad = jax.value_and_grad(f)(jnp.pi / 4)
# val  → sin(π/4) ≈ 0.7071
# grad → cos(π/4) ≈ 0.7071
val, grad

(Array(0.70710677, dtype=float32, weak_type=True),
 Array(0.70710677, dtype=float32, weak_type=True))

### ✅ Exercise 44 — Gradient w.r.t. Specific Argument

Real models have many parameters (W, b, x, ...). `argnums` lets you differentiate w.r.t. only what you want.

In [4]:
# Function with multiple arguments
f = lambda x, y: x**2 + y**3
grad_wrt_y = jax.grad(f, argnums=1)  # Differentiate w.r.t. second argument (y)
grad_wrt_y(2.0, 3.0)  # f'_y = 3y² → 27.0

Array(27., dtype=float32, weak_type=True)

### ✅ Exercise 45 — Gradient Over a Vector

When input is a vector, `grad` returns a vector of partial derivatives (the gradient ∇f). This is exactly how gradient descent updates work.

In [5]:
# Vector input - gradient returns vector of partial derivatives
w = jnp.array([1.0, 2.0, 3.0])
jax.grad(lambda w: jnp.dot(w, w))(w)  # → 2w = [2., 4., 6.]

Array([2., 4., 6.], dtype=float32)

### ✅ Exercise 46 — Second Derivative

Apply `grad` twice to get f''(x). Used in Newton's method and second-order optimizers.

In [6]:
# Second derivative by applying grad twice
f        = lambda x: x ** 4
df       = jax.grad(f)      # First derivative
d2f      = jax.grad(df)    # Second derivative
d2f(2.0)  # f''(x) = 12x² → 48.0

Array(48., dtype=float32, weak_type=True)

### ✅ Exercise 47 — JIT + Grad

`grad` is a Python transformation that produces a new Python function. That function can itself be JIT compiled for maximum performance.

### ✅ Exercise 48 — Jacobian of a Vector Function

`grad` only works for scalar outputs. When your function outputs a **vector**, you need the **Jacobian matrix** — all partial derivatives of all outputs w.r.t. all inputs.

In [7]:
# Vector function returning multiple outputs
def f(x):
    return jnp.array([x[0]**2, x[0]*x[1], x[1]**3])

x = jnp.array([2.0, 3.0])
J = jax.jacobian(f)(x)
# J[i, j] = ∂f_i / ∂x_j
# → [[4., 0.],  # ∂f₁/∂x₁, ∂f₁/∂x₂
#    [3., 2.],  # ∂f₂/∂x₁, ∂f₂/∂x₂  
#    [0., 27.]] # ∂f₃/∂x₁, ∂f₃/∂x₂
print(J)

[[ 4.  0.]
 [ 3.  2.]
 [ 0. 27.]]


### ✅ Exercise 49 — Gradient with Reduction Over Matrix

Works identically to vector case — JAX treats the matrix as a flat parameter space. This is how weight matrices in neural networks receive gradients.

**What "flat parameter space" means:**
- JAX ignores the 2D structure and treats a 3×3 matrix as 9 individual elements
- Computes gradient for each element separately: ∂f/∂wᵢⱼ
- Reshapes result back to original matrix shape
- Each weight in neural networks gets its own gradient for updates

In [8]:
# Matrix gradient - JAX treats matrix as flat parameter space
W = jnp.ones((3, 3))
f = lambda W: jnp.sum(W ** 2)  # Sum of squares
jax.grad(f)(W)
# f = Σ wᵢⱼ²  →  ∂f/∂wᵢⱼ = 2wᵢⱼ
# All ones → gradient is all 2.0s → (3,3) matrix of 2.0

Array([[2., 2., 2.],
       [2., 2., 2.],
       [2., 2., 2.]], dtype=float32)

### ✅ Exercise 50 — Stop Gradient

Sometimes we want to freeze part of a computation — treat it as a constant during differentiation.

In [9]:
# f(x) = x * stop_gradient(x + 1)
# Treating (x+1) as constant c:
# f(x) = c * x  →  f'(x) = c = (x+1)
# Without stop_gradient: f'(x) = 2x + 1

f_stopped = lambda x: x * jax.lax.stop_gradient(x + 1)
f_normal  = lambda x: x * (x + 1)

x = 3.0
print(jax.grad(f_stopped)(x))  # → x+1 = 4.0  (frozen term ignored)
print(jax.grad(f_normal)(x))   # → 2x+1 = 7.0

4.0
7.0
